# MSENet baseline vs. Adjacent vs. Bidirectional Cross-Attention (MiniImageNet)

One notebook, three independent experiments, one ZIP of results. **Run All** on a Kaggle GPU (T4) with Internet on.

| # | Model | Only difference |
|---|-------|-----------------|
| 1 | **Original MSENet** | none – baseline code copied verbatim from `baseline.ipynb` |
| 2 | **Adjacent cross-attention** | stage *i* (Query) attends to stage *i+1* (Key/Value), i = 1..4 |
| 3 | **Bidirectional cross-attention** | adjacent links **plus** the reverse links: stage *i+1* (Query) attends to stage *i* |

**Where cross-attention sits:** backbone stage → baseline self-attention (unchanged) → **cross-attention** → global average pooling → prototypes/distance (unchanged).

**Design decisions (all applied identically to support and query features)**
* Q/K/V use 1×1 convs. Q and K are projected to `d = C_query // 8` (same convention as the baseline `Self_Attn`); V is projected straight to the *query stage's* channel count so the update can be added to the query feature map.
* Scaled dot-product attention: `softmax(QKᵀ/√d)·V`.
* Spatial resolutions differ between stages (21, 21, 11, 6, 3 for 84 px input). Attention works on token sets, so keys/values stay at their **native resolution** (no lossy interpolation) and the output lands on the **query stage's own grid** – shapes match for the residual by construction (asserted in the sanity cell).
* Residual: `F_q' = F_q + γ · CrossAttn(F_q, F_kv)`, with one learnable scalar γ per link, initialised to the baseline's `gamma_init = 0.2` and trained with the baseline's γ learning rate. One shared module and γ per link serves both support and query.
* All K/V come from the **self-attention-refined features before any cross update** (parallel update, no cascade). Hence *Adjacent ⊂ Bidirectional* and the two differ only by the reverse links. In the bidirectional model a stage that is touched by two links (as Query in one pair, as Key/Value partner in the other) simply receives both residual updates, so the five stages / five stage weights / five prototype sets of the baseline are preserved.
* The deepest stage has no deeper partner, so it is only updated in the bidirectional model (by querying stage 4).

**Fairness controls:** identical seed and identical initial weights for all shared parameters (checked), identical training episodes and augmentation flips (RNG re-seeded before each training run), **identical fixed validation/test episodes** for all models, identical optimiser/scheduler/AMP/clipping, models trained independently from scratch.

**Evaluation protocol:** 5-way **5-shot** only (the baseline's training configuration: 30-way 5-shot, 15 queries). The best-validation checkpoint (as in the baseline) is tested on 600 fixed 5-way 5-shot episodes, identical for all three models.

In [ ]:
!pip install -q easyfsl gdown

In [ ]:
# ============================== Environment, imports, configuration ==============================
import os, sys, time, random, copy, gc, json, math, shutil, zipfile, warnings, platform, subprocess, tarfile, types
from pathlib import Path
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd
import torch
from torch import nn, optim
from torch.utils.data import DataLoader, Sampler
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torchvision.models import resnet18, ResNet18_Weights
from torchvision.models.feature_extraction import create_feature_extractor, get_graph_node_names
from easyfsl.samplers import TaskSampler
from tqdm.auto import tqdm
import multiprocessing
import matplotlib
import matplotlib.pyplot as plt

# Silence non-critical warnings only. Errors and exceptions are never suppressed.
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=DeprecationWarning)
os.environ['PYTHONWARNINGS'] = 'ignore'

# Force 'spawn' start method (same as baseline)
try:
    multiprocessing.set_start_method('spawn', force=True)
except RuntimeError:
    pass

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

SEED = 42
set_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
if device.type == 'cuda': print('GPU:', torch.cuda.get_device_name(0))
else: print('WARNING: no GPU detected - select a GPU accelerator in Kaggle (Settings > Accelerator).')

# ---------------- Paths ----------------
ON_KAGGLE = os.path.isdir('/kaggle/working')
WORK_DIR = Path('/kaggle/working') if ON_KAGGLE else Path('./working').resolve()
WORK_DIR.mkdir(parents=True, exist_ok=True)
# Large, re-creatable files go to /kaggle/temp so they do not bloat the notebook output.
TEMP_DIR = Path('/kaggle/temp') if Path('/kaggle').is_dir() else WORK_DIR
try:
    TEMP_DIR.mkdir(parents=True, exist_ok=True)
except OSError:
    TEMP_DIR = WORK_DIR
DOWNLOAD_DIR = TEMP_DIR / 'miniimagenet_archives'
DATA_DIR     = TEMP_DIR / 'miniimagenet'
RESULTS_DIR  = WORK_DIR / 'msenet_results'
ZIP_PATH     = WORK_DIR / 'msenet_cross_attention_results.zip'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# ---------------- Baseline experiment configuration (unchanged values from baseline.ipynb) ----------------
IMAGE_SIZE = 84
MAX_EPOCHS = 10
N_TRAINING_EPISODES = 360
N_VALIDATION_TASKS = 600
N_TEST_TASKS = 600
TRAIN_WAY = 30
TEST_WAY = 5
SHOT = 5
QUERY = 15
FEATURE_CHUNK_SIZE = 600
ATTENTION_CHUNK_SIZE = 600
DISTANCE_METRIC = 'euclidean'
COSINE_TEMP = 10.0
USE_AMP = True
STAGE_NODES   = ('maxpool', 'layer1', 'layer2', 'layer3', 'layer4')
STAGE_WEIGHTS = (1.0, 1.1, 1.2, 1.3, 1.4)
USE_IMAGENET_PRETRAINED = True
NUM_WORKERS = 0
gamma_init = 0.2
weights_lr = 0.005
backbone_lr = 1e-4
AMP = USE_AMP and device.type == 'cuda'

# ---------------- Settings added for the comparison (do not change the models) ----------------
EVAL_SHOTS = (5,)            # 5-shot only
EVAL_SEED = 2024             # seed that defines the fixed validation / test episodes
CROSS_GAMMA_INIT = gamma_init
BENCH_WARMUP, BENCH_TIMED = 5, 100      # inference-timing episodes
DIAG_TASKS = 30                         # episodes used for attention statistics
PRELOAD_DATASETS = True                 # decode images once into RAM (identical tensors, much faster)
PRELOAD_THREADS = min(8, os.cpu_count() or 4)
PRELOAD_CHUNK = 512                     # RAM note: all splits together use ~4.5 GB float32 (train 3.3 GB, val/test 0.6 GB each) - fine for Kaggle's ~29 GB

_norm = transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
train_transform = transforms.Compose([transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)), transforms.RandomHorizontalFlip(),
                                      transforms.ToTensor(), _norm])
eval_transform = transforms.Compose([transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)), transforms.ToTensor(), _norm])
print('Configuration ready.')

In [ ]:
# ============================== Dataset download + extraction (MiniImageNet from Google Drive, via gdown) ==============================
DRIVE_FOLDER = "https://drive.google.com/drive/folders/17a09kkqVivZQFggCw9I_YboJ23tcexNM"
SPLITS = ("train", "val", "test")
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

def resolve_split_dir(split):
    """Directory that directly contains the class folders of a split (handles train/train and flat layouts)."""
    base = DATA_DIR / split
    for cand in (base / split, base):
        if cand.is_dir():
            subs = [d for d in cand.iterdir() if d.is_dir() and not d.name.startswith(('.', '_'))]
            if len(subs) >= 2:
                return cand
    raise FileNotFoundError(f"No class folders found for split '{split}' under {base}")

def split_ready(split):
    try:
        resolve_split_dir(split); return True
    except FileNotFoundError:
        return False

def archives_ready():
    return all(list(DOWNLOAD_DIR.rglob(f"{s}.tar")) for s in SPLITS)

if all(split_ready(s) for s in SPLITS):
    print("Dataset already extracted - skipping download.")
else:
    for attempt in range(1, 4):
        if archives_ready():
            break
        print(f"Downloading Drive folder with gdown (attempt {attempt}/3) ...")
        !gdown --folder "{DRIVE_FOLDER}" -O "{DOWNLOAD_DIR}"
    if not archives_ready():
        raise FileNotFoundError("train.tar / val.tar / test.tar were not downloaded. Make sure Internet is enabled in the "
                                "Kaggle notebook settings and the Drive folder is publicly accessible, then re-run.")
    for p in sorted(DOWNLOAD_DIR.rglob("*.tar")):
        print(f"  {p.name}: {p.stat().st_size / 1e6:.1f} MB")

    for split in SPLITS:
        if split_ready(split):
            continue
        archive = str(next(DOWNLOAD_DIR.rglob(f"{split}.tar")))
        out_dir = str(DATA_DIR / split)
        os.makedirs(out_dir, exist_ok=True)
        print(f"Extracting {archive} -> {out_dir} ...")
        !tar -xf "{archive}" -C "{out_dir}"
        if not split_ready(split):
            raise RuntimeError(f"Extraction of {split}.tar did not produce class folders under {out_dir}")
        os.remove(archive)                      # free disk space (archives are not needed after extraction)

TRAIN_DIR, VAL_DIR, TEST_DIR = (str(resolve_split_dir(s)) for s in SPLITS)
for split, path in zip(SPLITS, (TRAIN_DIR, VAL_DIR, TEST_DIR)):
    classes = [d for d in Path(path).iterdir() if d.is_dir()]
    print(f"{split}: {path} | class folders={len(classes)} | examples={[d.name for d in classes[:3]]}")

In [ ]:
# Fast episode sampler compatibility patch for EasyFSL / newer PyTorch
# The patch uses precomputed labels; it never loads image files to discover labels.
def _patched_task_sampler_init(self, dataset, n_way, n_shot, n_query, n_tasks):
    Sampler.__init__(self)
    self.n_way, self.n_shot, self.n_query, self.n_tasks = n_way, n_shot, n_query, n_tasks
    self.items_per_label = defaultdict(list)
    for item, label in enumerate(dataset.get_labels()):
        self.items_per_label[int(label)].append(item)
    if len(self.items_per_label) < n_way:
        raise ValueError(f'Requested {n_way}-way episodes but dataset has only {len(self.items_per_label)} classes.')
    required = n_shot + n_query
    too_small = [label for label, items in self.items_per_label.items() if len(items) < required]
    if too_small:
        raise ValueError(f'Classes have fewer than {required} images: {too_small[:5]}')

TaskSampler.__init__ = _patched_task_sampler_init
print('TaskSampler patched; labels are read from metadata only.')

In [ ]:
# Evaluation helpers
@torch.no_grad()
def evaluate_on_one_task(model, support_images, support_labels, query_images, query_labels):
    scores = model(support_images.to(device), support_labels.to(device), query_images.to(device))
    predictions = scores.argmax(dim=1)
    labels = query_labels.to(device)
    return (predictions == labels).sum().item(), labels.numel()

@torch.no_grad()
def evaluate(data_loader, model, desc='Evaluating'):
    model.eval()
    correct_total, prediction_total, n_tasks = 0, 0, 0
    for support_images, support_labels, query_images, query_labels, _ in tqdm(data_loader, total=len(data_loader), desc=desc, leave=False):
        correct, total = evaluate_on_one_task(model, support_images, support_labels, query_images, query_labels)
        correct_total += correct
        prediction_total += total
        n_tasks += 1
    if prediction_total == 0:
        raise RuntimeError('Evaluation loader produced no query examples.')
    if n_tasks != len(data_loader):
        raise RuntimeError(f'Evaluated {n_tasks} tasks but loader promised {len(data_loader)}.')
    return 100.0 * correct_total / prediction_total

In [ ]:
# Multi-scale attention model
class Self_Attn(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        if in_dim < 8:
            raise ValueError(f'in_dim must be >= 8, got {in_dim}')
        self.query_conv = nn.Conv2d(in_dim, in_dim // 8, kernel_size=1)
        self.key_conv = nn.Conv2d(in_dim, in_dim // 8, kernel_size=1)
        self.value_conv = nn.Conv2d(in_dim, in_dim, kernel_size=1)

    def forward(self, x, gamma):
        b, c, h, w = x.shape
        n = h * w
        q = self.query_conv(x).reshape(b, -1, n).transpose(1, 2)  # B,N,C/8
        k = self.key_conv(x).reshape(b, -1, n)                   # B,C/8,N
        attention = torch.softmax(torch.bmm(q, k), dim=-1)        # B,N,N
        v = self.value_conv(x).reshape(b, c, n)                   # B,C,N
        out = torch.bmm(v, attention.transpose(1, 2)).reshape(b, c, h, w)
        return gamma * out + x

class PrototypicalNetworks(nn.Module):
    def __init__(self, backbone, image_size, stage_nodes=STAGE_NODES, weight=STAGE_WEIGHTS,
                 weightlearnable=True, gammalearnable=True, gamma1=0.2, gamma2=0.2,
                 feature_chunk_size=600, attention_chunk_size=600,
                 distance_metric='cosine', cosine_temp=10.0):
        super().__init__()
        if len(weight) != len(stage_nodes):
            raise ValueError(f'{len(weight)} weights given for {len(stage_nodes)} stages.')
        self.weightlearnable = weightlearnable
        self.gammalearnable = gammalearnable
        self.feature_chunk_size = feature_chunk_size
        self.attention_chunk_size = attention_chunk_size
        self.n_stages = len(stage_nodes)
        self.distance_metric = distance_metric
        
        # Learnable temperature scalar for cosine distance scaling
        self.cosine_temp = nn.Parameter(torch.tensor(float(cosine_temp)), requires_grad=True)

        self.stage_weights = nn.Parameter(torch.tensor(list(weight), dtype=torch.float32),
                                          requires_grad=weightlearnable)
        self.gamma1_l = nn.Parameter(torch.tensor(float(gamma1)), requires_grad=gammalearnable)
        self.gamma2_l = nn.Parameter(torch.tensor(float(gamma2)), requires_grad=gammalearnable)

        graph_nodes, _ = get_graph_node_names(backbone)
        return_nodes = {}
        for i, name in enumerate(stage_nodes):
            matches = [n for n in graph_nodes if n == name or n.startswith(name + '.')]
            if not matches:
                raise ValueError(f'Stage {name!r} not found in backbone graph.')
            return_nodes[matches[-1]] = f'stage_{i}'
        print('Stage nodes:', return_nodes)

        self.feature_extractor = create_feature_extractor(backbone, return_nodes=return_nodes)

        was_training = self.feature_extractor.training
        self.feature_extractor.eval()
        with torch.no_grad():
            dummy = self.feature_extractor(torch.zeros(1, 3, image_size, image_size))
        self.feature_extractor.train(was_training)
        channels = [dummy[f'stage_{i}'].shape[1] for i in range(self.n_stages)]
        print('Selected feature channels:', channels,
              '| spatial sizes:', [tuple(dummy[f'stage_{i}'].shape[2:]) for i in range(self.n_stages)])

        self.attention_modules = nn.ModuleList([Self_Attn(c) for c in channels])
        self.global_avg_pool = nn.AdaptiveAvgPool2d(1)

    def _extract_in_chunks(self, images):
        # Fast path: if batch fits in chunk_size, skip Python loop & dict rebuilding!
        if images.size(0) <= self.feature_chunk_size:
            return self.feature_extractor(images)
        collected = None
        for chunk in images.split(self.feature_chunk_size, dim=0):
            feats = self.feature_extractor(chunk)
            if collected is None:
                collected = {key: [value] for key, value in feats.items()}
            else:
                for key, value in feats.items(): collected[key].append(value)
        return {key: torch.cat(values, dim=0) for key, values in collected.items()}

    def _attend_and_pool(self, features, stage_idx, gamma):
        module = self.attention_modules[stage_idx]
        # Fast path: skip chunking loop when batch size <= attention_chunk_size
        if features.size(0) <= self.attention_chunk_size:
            return self.global_avg_pool(module(features, gamma)).flatten(1)
        pooled = []
        for chunk in features.split(self.attention_chunk_size, dim=0):
            pooled.append(self.global_avg_pool(module(chunk, gamma)).flatten(1))
        return torch.cat(pooled, dim=0)

    def _stage_distance(self, support_features, query_features, support_labels, stage_idx):
        z_support = self._attend_and_pool(support_features, stage_idx, self.gamma1_l)
        z_query = self._attend_and_pool(query_features, stage_idx, self.gamma2_l)
        
        # Vectorized prototype computation (faster than Python list comprehension over 30 labels)
        labels = torch.unique(support_labels, sorted=True)
        prototypes = torch.stack([z_support[support_labels == label].mean(0) for label in labels])

        if self.distance_metric == 'euclidean':
            return torch.cdist(z_query, prototypes, p=2)
            
        elif self.distance_metric == 'cosine':
            # L2-normalize query and prototype vectors onto unit hypersphere
            z_q_norm = nn.functional.normalize(z_query, p=2, dim=-1)
            proto_norm = nn.functional.normalize(prototypes, p=2, dim=-1)
            # Cosine distance = (1 - cosine_similarity) scaled by temperature
            cos_sim = torch.mm(z_q_norm, proto_norm.t())
            return self.cosine_temp * (1.0 - cos_sim)
            
        elif self.distance_metric == 'hybrid':
            # Combines L2-normalized Euclidean distance and Cosine distance
            euc_dist = torch.cdist(z_query, prototypes, p=2)
            z_q_norm = nn.functional.normalize(z_query, p=2, dim=-1)
            proto_norm = nn.functional.normalize(prototypes, p=2, dim=-1)
            cos_dist = self.cosine_temp * (1.0 - torch.mm(z_q_norm, proto_norm.t()))
            return 0.5 * euc_dist + 0.5 * cos_dist
        else:
            raise ValueError(f"Unknown distance_metric: {self.distance_metric}")

    def forward(self, support_images, support_labels, query_images):
        support_feats = self._extract_in_chunks(support_images)
        query_feats = self._extract_in_chunks(query_images)
        distances = torch.stack([
            self._stage_distance(support_feats[f'stage_{i}'], query_feats[f'stage_{i}'], support_labels, i)
            for i in range(self.n_stages)
        ])
        return -(self.stage_weights.view(-1, 1, 1) * distances).sum(dim=0)

print('Model class defined.')

In [ ]:
# ============================== Datasets and episode loaders ==============================
class PreloadedDataset(torch.utils.data.Dataset):
    """Decodes every image ONCE through the baseline's deterministic transform (Resize -> ToTensor -> Normalize)
    and keeps the tensors in RAM. The baseline's random horizontal flip is applied on the fly for training.
    Resize/ToTensor/Normalize commute with a flip, so the tensors are identical to the baseline pipeline."""
    def __init__(self, folder_ds, augment=False, desc=''):
        self.classes = folder_ds.classes
        self.targets = list(folder_ds.targets)
        self.augment = augment
        self.flip = transforms.RandomHorizontalFlip()
        n = len(folder_ds)
        self.images = torch.empty(n, 3, IMAGE_SIZE, IMAGE_SIZE)
        def load(i):
            path, _ = folder_ds.samples[i]
            return i, folder_ds.transform(folder_ds.loader(path))
        # Decode in bounded chunks so at most PRELOAD_CHUNK decoded tensors are ever in flight (keeps RAM low).
        bar = tqdm(total=n, desc=f'Pre-loading {desc}', leave=False)
        with ThreadPoolExecutor(max_workers=PRELOAD_THREADS) as ex:
            for start in range(0, n, PRELOAD_CHUNK):
                for i, x in ex.map(load, range(start, min(start + PRELOAD_CHUNK, n))):
                    self.images[i] = x
                bar.update(min(start + PRELOAD_CHUNK, n) - start)
        bar.close()
    def __len__(self): return len(self.targets)
    def __getitem__(self, i):
        x = self.images[i]
        if self.augment: x = self.flip(x)
        return x, self.targets[i]
    def get_labels(self): return self.targets

t0 = time.time()
if PRELOAD_DATASETS:
    train_set = PreloadedDataset(ImageFolder(TRAIN_DIR, transform=eval_transform), augment=True,  desc='train')
    val_set   = PreloadedDataset(ImageFolder(VAL_DIR,   transform=eval_transform), augment=False, desc='val')
    test_set  = PreloadedDataset(ImageFolder(TEST_DIR,  transform=eval_transform), augment=False, desc='test')
else:
    train_set = ImageFolder(TRAIN_DIR, transform=train_transform)
    val_set   = ImageFolder(VAL_DIR,   transform=eval_transform)
    test_set  = ImageFolder(TEST_DIR,  transform=eval_transform)
    for ds in (train_set, val_set, test_set):
        ds.get_labels = types.MethodType(lambda self: self.targets, ds)

print(f"Classes: train={len(train_set.classes)}, val={len(val_set.classes)}, test={len(test_set.classes)}")
print(f"Images:  train={len(train_set)}, val={len(val_set)}, test={len(test_set)}   (prepared in {time.time()-t0:.0f}s)")
assert len(train_set.classes) >= TRAIN_WAY, "Not enough training classes for TRAIN_WAY."
assert len(val_set.classes) >= TEST_WAY and len(test_set.classes) >= TEST_WAY

def make_loader(dataset, n_way, n_tasks):          # baseline loader (fresh random episodes every epoch)
    sampler = TaskSampler(dataset, n_way=n_way, n_shot=SHOT, n_query=QUERY, n_tasks=n_tasks)
    loader = DataLoader(dataset, batch_sampler=sampler, num_workers=NUM_WORKERS,
                        pin_memory=(device.type == "cuda"), collate_fn=sampler.episodic_collate_fn)
    assert len(loader) == n_tasks, f"Loader has {len(loader)} tasks, expected {n_tasks}."
    return loader

class ReplaySampler(Sampler):
    """Replays a pre-generated list of episodes so every model is evaluated on exactly the same tasks."""
    def __init__(self, batches): self.batches = batches
    def __iter__(self): return iter(self.batches)
    def __len__(self): return len(self.batches)

def make_fixed_loader(dataset, n_way, n_shot, n_tasks, seed):
    sampler = TaskSampler(dataset, n_way=n_way, n_shot=n_shot, n_query=QUERY, n_tasks=n_tasks)
    state = random.getstate(); random.seed(seed)
    batches = [list(b) for b in sampler]
    random.setstate(state)
    loader = DataLoader(dataset, batch_sampler=ReplaySampler(batches), num_workers=NUM_WORKERS,
                        pin_memory=(device.type == "cuda"), collate_fn=sampler.episodic_collate_fn)
    assert len(loader) == n_tasks
    return loader

train_loader = make_loader(train_set, TRAIN_WAY, N_TRAINING_EPISODES)
val_loader   = make_fixed_loader(val_set, TEST_WAY, SHOT, N_VALIDATION_TASKS, EVAL_SEED)           # 5-shot validation
test_loaders = {shot: make_fixed_loader(test_set, TEST_WAY, shot, N_TEST_TASKS, EVAL_SEED + shot) for shot in EVAL_SHOTS}
print(f"Episodes: train/epoch={len(train_loader)}, val={len(val_loader)}, test={{{', '.join(f'{k}-shot: {len(v)}' for k, v in test_loaders.items())}}}")

def check_episode(loader, way, shot):
    s_img, s_lab, q_img, q_lab, _ = next(iter(loader))
    assert s_img.shape == (way * shot, 3, IMAGE_SIZE, IMAGE_SIZE), s_img.shape
    assert q_img.shape == (way * QUERY, 3, IMAGE_SIZE, IMAGE_SIZE), q_img.shape
    assert len(set(s_lab.tolist())) == way
check_episode(train_loader, TRAIN_WAY, SHOT); check_episode(val_loader, TEST_WAY, SHOT)
for shot, ld in test_loaders.items(): check_episode(ld, TEST_WAY, shot)
print('Episode shapes OK.')

In [ ]:
# ============================== Cross-attention modules (the ONLY addition to the baseline) ==============================
class CrossAttn(nn.Module):
    """Scaled dot-product cross-attention between two feature maps of (possibly) different resolution.
    Query tokens come from `x_q` (B,Cq,Hq,Wq); keys/values come from `x_kv` (B,Ckv,Hk,Wk).
    1x1 convs project Q,K to d = Cq//8 and V to Cq channels. forward() returns the residual UPDATE gamma*out
    on the query grid (B,Cq,Hq,Wq); the caller adds it to the query features (residual connection)."""
    def __init__(self, q_dim, kv_dim, gamma_init=0.2, label=''):
        super().__init__()
        self.d = max(8, q_dim // 8)
        self.scale = self.d ** -0.5
        self.query_conv = nn.Conv2d(q_dim, self.d, kernel_size=1)
        self.key_conv   = nn.Conv2d(kv_dim, self.d, kernel_size=1)
        self.value_conv = nn.Conv2d(kv_dim, q_dim, kernel_size=1)
        self.gamma = nn.Parameter(torch.tensor(float(gamma_init)))
        self.label = label
        self.record = False
        self.reset_stats()

    def reset_stats(self):
        self._stats = {'calls': 0, 'entropy': 0.0, 'max_attn': 0.0, 'update_ratio': 0.0, 'nq': 0, 'nk': 0}

    def forward(self, x_q, x_kv):
        b, c, hq, wq = x_q.shape
        nq, nk = hq * wq, x_kv.shape[2] * x_kv.shape[3]
        q = self.query_conv(x_q).reshape(b, self.d, nq).transpose(1, 2)    # B,Nq,d
        k = self.key_conv(x_kv).reshape(b, self.d, nk)                     # B,d,Nk
        v = self.value_conv(x_kv).reshape(b, c, nk)                        # B,C,Nk
        attn = torch.softmax(torch.bmm(q, k) * self.scale, dim=-1)         # B,Nq,Nk  (scaled dot-product)
        out = torch.bmm(v, attn.transpose(1, 2)).reshape(b, c, hq, wq)     # B,C,Hq,Wq  (query grid)
        delta = self.gamma * out
        if self.record:
            with torch.no_grad():
                a = attn.float()
                st = self._stats
                st['calls'] += 1
                st['entropy'] += (-(a * torch.log(a.clamp_min(1e-12))).sum(-1).mean() / math.log(max(nk, 2))).item()
                st['max_attn'] += a.max(-1).values.mean().item()
                st['update_ratio'] += (delta.float().norm() / x_q.float().norm().clamp_min(1e-12)).item()
                st['nq'], st['nk'] = nq, nk
        return delta

class CrossAttnPrototypicalNetworks(PrototypicalNetworks):
    """Baseline MSENet + cross-attention between consecutive stages, inserted after the (unchanged) self-attention
    and before global average pooling. cross_mode: 'adjacent' (shallow queries deeper) or
    'bidirectional' (adjacent links + deeper queries shallow). All K/V use the self-attention-refined features
    from BEFORE any cross update, so the two modes differ only by the reverse links."""
    def __init__(self, *args, cross_mode='adjacent', cross_gamma_init=0.2, **kwargs):
        super().__init__(*args, **kwargs)
        if cross_mode not in ('adjacent', 'bidirectional'):
            raise ValueError(f'Unknown cross_mode: {cross_mode}')
        self.cross_mode = cross_mode
        ch = [m.query_conv.in_channels for m in self.attention_modules]
        n = self.n_stages
        # cross_fwd[i]: stage i (Query) attends to stage i+1 (Key/Value)
        self.cross_fwd = nn.ModuleList([CrossAttn(ch[i], ch[i + 1], cross_gamma_init, f'Stage{i+1} queries Stage{i+2}')
                                        for i in range(n - 1)])
        # cross_bwd[i]: stage i+1 (Query) attends to stage i (Key/Value)
        self.cross_bwd = nn.ModuleList([CrossAttn(ch[i + 1], ch[i], cross_gamma_init, f'Stage{i+2} queries Stage{i+1}')
                                        for i in range(n - 1)]) if cross_mode == 'bidirectional' else nn.ModuleList()

    def _refine_chunk(self, feats, gamma):
        n = self.n_stages
        att = [self.attention_modules[i](feats[i], gamma) for i in range(n)]      # baseline self-attention
        pooled = []
        for j in range(n):
            out = att[j]
            if j < n - 1:
                out = out + self.cross_fwd[j](att[j], att[j + 1])                 # shallower queries deeper
            if self.cross_mode == 'bidirectional' and j > 0:
                out = out + self.cross_bwd[j - 1](att[j], att[j - 1])             # deeper queries shallower
            assert out.shape == att[j].shape                                      # residual keeps the stage's own grid
            pooled.append(self.global_avg_pool(out).flatten(1))
        return pooled

    def _encode(self, feats_dict, gamma):
        feats = [feats_dict[f'stage_{i}'] for i in range(self.n_stages)]
        bsz = feats[0].size(0)
        if bsz <= self.attention_chunk_size:
            return self._refine_chunk(feats, gamma)
        parts = [self._refine_chunk(list(c), gamma) for c in zip(*[f.split(self.attention_chunk_size, dim=0) for f in feats])]
        return [torch.cat([p[i] for p in parts], dim=0) for i in range(self.n_stages)]

    def _prototype_distance(self, z_support, z_query, support_labels):           # identical to baseline _stage_distance
        labels = torch.unique(support_labels, sorted=True)
        prototypes = torch.stack([z_support[support_labels == label].mean(0) for label in labels])
        if self.distance_metric == 'euclidean':
            return torch.cdist(z_query, prototypes, p=2)
        elif self.distance_metric == 'cosine':
            z_q_norm = nn.functional.normalize(z_query, p=2, dim=-1)
            proto_norm = nn.functional.normalize(prototypes, p=2, dim=-1)
            return self.cosine_temp * (1.0 - torch.mm(z_q_norm, proto_norm.t()))
        elif self.distance_metric == 'hybrid':
            euc_dist = torch.cdist(z_query, prototypes, p=2)
            z_q_norm = nn.functional.normalize(z_query, p=2, dim=-1)
            proto_norm = nn.functional.normalize(prototypes, p=2, dim=-1)
            cos_dist = self.cosine_temp * (1.0 - torch.mm(z_q_norm, proto_norm.t()))
            return 0.5 * euc_dist + 0.5 * cos_dist
        raise ValueError(f"Unknown distance_metric: {self.distance_metric}")

    def forward(self, support_images, support_labels, query_images):
        support_feats = self._extract_in_chunks(support_images)
        query_feats = self._extract_in_chunks(query_images)
        z_support = self._encode(support_feats, self.gamma1_l)     # same cross-attention modules for support ...
        z_query = self._encode(query_feats, self.gamma2_l)         # ... and query
        distances = torch.stack([self._prototype_distance(z_support[i], z_query[i], support_labels)
                                 for i in range(self.n_stages)])
        return -(self.stage_weights.view(-1, 1, 1) * distances).sum(dim=0)

print('Cross-attention classes defined.')

In [ ]:
# ============================== Experiment utilities ==============================
EXPERIMENTS = [
    dict(key='baseline',      name='Original MSENet (baseline)', short='Baseline',         cls=PrototypicalNetworks,           extra={}),
    dict(key='adjacent',      name='Adjacent Cross-Attention',   short='Adjacent CA',      cls=CrossAttnPrototypicalNetworks,  extra=dict(cross_mode='adjacent',      cross_gamma_init=CROSS_GAMMA_INIT)),
    dict(key='bidirectional', name='Bidirectional Cross-Attention', short='Bidirectional CA', cls=CrossAttnPrototypicalNetworks, extra=dict(cross_mode='bidirectional', cross_gamma_init=CROSS_GAMMA_INIT)),
]
COLORS = {'baseline': '#4C72B0', 'adjacent': '#DD8452', 'bidirectional': '#55A868'}

def sync():
    if device.type == 'cuda': torch.cuda.synchronize()

def _pretrained_backbone_state():
    bb = None
    if USE_IMAGENET_PRETRAINED:
        try:
            bb = resnet18(weights=ResNet18_Weights.DEFAULT)
        except Exception as exc:
            print('Could not load torchvision pretrained weights; using random initialization:', exc)
    if bb is None:
        bb = resnet18(weights=None)
    bb.fc = nn.Identity()
    return bb.state_dict()

BACKBONE_STATE = _pretrained_backbone_state()      # loaded once -> every model starts from the same backbone

def build_model(exp):
    set_seed(SEED)                                  # identical RNG state -> identical init of all shared parameters
    backbone = resnet18(weights=None); backbone.fc = nn.Identity()
    backbone.load_state_dict(BACKBONE_STATE)
    return exp['cls'](
        backbone=backbone, image_size=IMAGE_SIZE, stage_nodes=STAGE_NODES, weight=STAGE_WEIGHTS,
        weightlearnable=True, gammalearnable=True, gamma1=gamma_init, gamma2=gamma_init,
        feature_chunk_size=FEATURE_CHUNK_SIZE, attention_chunk_size=ATTENTION_CHUNK_SIZE,
        distance_metric=DISTANCE_METRIC, cosine_temp=COSINE_TEMP, **exp['extra']).to(device)

def cross_modules(model):
    return [(n, m) for n, m in model.named_modules() if isinstance(m, CrossAttn)]

def count_params(model):
    numel = lambda ps: sum(p.numel() for p in ps)
    cross = cross_modules(model)
    cross_conv = numel([p for _, m in cross for c in (m.query_conv, m.key_conv, m.value_conv) for p in c.parameters()])
    cross_gate = len(cross)
    scalars = model.stage_weights.numel() + 2 + model.cosine_temp.numel()
    return dict(total=numel(model.parameters()), trainable=numel(p for p in model.parameters() if p.requires_grad),
                backbone=numel(model.feature_extractor.parameters()), self_attention=numel(model.attention_modules.parameters()),
                cross_attention_convs=cross_conv, cross_attention_gates=cross_gate, scalars=scalars)

def learned_params(model):
    d = {f'stage_weight_{i+1}': float(w) for i, w in enumerate(model.stage_weights.detach().cpu().tolist())}
    d['gamma1_support_selfattn'] = float(model.gamma1_l.item())
    d['gamma2_query_selfattn'] = float(model.gamma2_l.item())
    d['cosine_temp (unused for euclidean)'] = float(model.cosine_temp.item())
    for _, m in cross_modules(model):
        d[f'cross_gate[{m.label}]'] = float(m.gamma.item())
    return d

def conv_summary(conv):
    w = conv.weight.detach().float()
    return dict(shape=list(w.shape), fro_norm=float(w.norm()), mean_abs=float(w.abs().mean()), std=float(w.std()),
                bias_norm=float(conv.bias.detach().float().norm()))

@torch.no_grad()
def evaluate_per_task(loader, model, desc='Testing'):
    model.eval()
    accs = []
    for s_img, s_lab, q_img, q_lab, _ in tqdm(loader, total=len(loader), desc=desc, leave=False):
        correct, total = evaluate_on_one_task(model, s_img, s_lab, q_img, q_lab)
        accs.append(100.0 * correct / total)
    return np.array(accs)

@torch.no_grad()
def evaluate_with_loss(loader, model, desc='Validating'):
    """Same forward/accuracy as the baseline `evaluate` (fp32, eval mode) but also returns the mean cross-entropy loss."""
    model.eval()
    correct = total = n = 0; loss_sum = 0.0
    for s_img, s_lab, q_img, q_lab, _ in tqdm(loader, total=len(loader), desc=desc, leave=False):
        scores = model(s_img.to(device), s_lab.to(device), q_img.to(device))
        labels = q_lab.to(device)
        loss_sum += nn.functional.cross_entropy(scores, labels).item(); n += 1
        correct += (scores.argmax(1) == labels).sum().item(); total += labels.numel()
    if total == 0: raise RuntimeError('Evaluation loader produced no query examples.')
    return 100.0 * correct / total, loss_sum / n

@torch.no_grad()
def benchmark_inference(model, loader):
    """Pure forward latency (fp32, eval mode, data already on the GPU) over fixed test episodes."""
    model.eval()
    tasks = []
    for s, sl, q, ql, _ in loader:
        tasks.append((s.to(device), sl.to(device), q.to(device)))
        if len(tasks) >= BENCH_WARMUP + BENCH_TIMED: break
    n_warm = min(BENCH_WARMUP, max(len(tasks) - 1, 0))
    for t in tasks[:n_warm]: model(*t)
    if device.type == 'cuda':
        sync(); torch.cuda.reset_peak_memory_stats()
    times = []
    for t in tasks[n_warm:]:
        sync(); t0 = time.perf_counter(); model(*t); sync(); times.append(time.perf_counter() - t0)
    n_query = tasks[0][2].shape[0]
    peak = torch.cuda.max_memory_allocated() / 2**20 if device.type == 'cuda' else float('nan')
    return dict(ms_per_episode_mean=1000 * float(np.mean(times)), ms_per_episode_median=1000 * float(np.median(times)),
                ms_per_query_image=1000 * float(np.mean(times)) / n_query, query_images_per_s=n_query / float(np.mean(times)),
                peak_gpu_mem_mb=peak, n_timed_episodes=len(times))

@torch.no_grad()
def measure_gflops(model, loader):
    try:
        from torch.utils.flop_counter import FlopCounterMode
        model.eval()
        s, sl, q, ql, _ = next(iter(loader))
        with FlopCounterMode(display=False) as fc:
            model(s.to(device), sl.to(device), q.to(device))
        return fc.get_total_flops() / 1e9
    except Exception as exc:                        # optional metric: report loudly, do not abort the whole run
        print('FLOP counting failed (reported as NaN):', repr(exc))
        return float('nan')

@torch.no_grad()
def collect_cross_diagnostics(model, loader):
    model.eval()
    mods = cross_modules(model)
    for _, m in mods: m.reset_stats(); m.record = True
    for i, (s, sl, q, ql, _) in enumerate(loader):
        if i >= DIAG_TASKS: break
        model(s.to(device), sl.to(device), q.to(device))
    rows = []
    for name, m in mods:
        m.record = False
        st = m._stats; c = max(st['calls'], 1)
        rows.append(dict(module=name, link=m.label, q_tokens=st['nq'], kv_tokens=st['nk'], d_qk=m.d, gamma=float(m.gamma.item()),
                         attn_entropy_normalized=st['entropy'] / c, attn_mean_max_weight=st['max_attn'] / c,
                         uniform_max_weight=1.0 / max(st['nk'], 1), update_ratio_norm=st['update_ratio'] / c,
                         params=sum(p.numel() for p in m.parameters()),
                         q_conv=conv_summary(m.query_conv), k_conv=conv_summary(m.key_conv), v_conv=conv_summary(m.value_conv)))
    return rows

def train_model(model, exp, outer_desc):
    """Training loop copied from the baseline (same optimiser, groups, scheduler, AMP, clipping, validation, best-val selection);
    additions: timing, history logging and keeping the best weights in memory (no .pth written)."""
    special = [model.stage_weights, model.gamma1_l, model.gamma2_l, model.cosine_temp]
    cross_gammas = [m.gamma for _, m in cross_modules(model)]          # [] for the baseline
    special_ids = {id(p) for p in special + cross_gammas}
    regular_params = [p for p in model.parameters() if id(p) not in special_ids]
    param_groups = [{'params': regular_params, 'lr': backbone_lr},
                    {'params': [model.cosine_temp], 'lr': weights_lr}]
    if model.weightlearnable: param_groups.append({'params': [model.stage_weights], 'lr': weights_lr})
    if model.gammalearnable:  param_groups.append({'params': [model.gamma1_l, model.gamma2_l] + cross_gammas, 'lr': weights_lr})
    optimizer = optim.Adam(param_groups)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS, eta_min=1e-6)
    scaler = torch.amp.GradScaler('cuda', enabled=AMP)
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    criterion = nn.CrossEntropyLoss()

    best_val_acc, best_epoch, best_state = -1.0, -1, None
    history = []
    if device.type == 'cuda': torch.cuda.reset_peak_memory_stats()
    set_seed(SEED)                              # same training episodes and flips for every model
    train_time = val_time = 0.0
    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        loss_sum, n_episodes, train_correct, train_total = 0.0, 0, 0, 0
        sync(); t_ep = time.perf_counter()
        progress = tqdm(train_loader, total=len(train_loader), desc=f'{outer_desc} | epoch {epoch}/{MAX_EPOCHS}', leave=False)
        for support_images, support_labels, query_images, query_labels, _ in progress:
            optimizer.zero_grad(set_to_none=True)
            labels_dev = query_labels.to(device)
            with torch.amp.autocast('cuda', enabled=AMP):
                scores = model(support_images.to(device, non_blocking=True), support_labels.to(device),
                               query_images.to(device, non_blocking=True))
                loss = criterion(scores, labels_dev)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(trainable_params, max_norm=5.0)
            scaler.step(optimizer)
            scaler.update()
            loss_sum += loss.item(); n_episodes += 1
            train_correct += (scores.argmax(1) == labels_dev).sum().item(); train_total += labels_dev.numel()
            progress.set_postfix(loss=f'{loss.item():.4f}')
        if n_episodes != N_TRAINING_EPISODES:
            raise RuntimeError(f'Ran {n_episodes} episodes, config says {N_TRAINING_EPISODES}.')
        scheduler.step()
        sync(); ep_train = time.perf_counter() - t_ep
        t_val = time.perf_counter()
        val_acc, val_loss = evaluate_with_loss(val_loader, model, desc=f'Validating epoch {epoch}')
        sync(); ep_val = time.perf_counter() - t_val
        train_time += ep_train; val_time += ep_val
        train_loss = loss_sum / n_episodes
        train_acc = 100.0 * train_correct / train_total      # running accuracy over the epoch's training episodes
        row = dict(model=exp['key'], epoch=epoch, train_loss=train_loss, train_acc=train_acc, val_loss=val_loss, val_acc=val_acc,
                   train_time_s=ep_train, val_time_s=ep_val, **learned_params(model))
        history.append(row)
        print(f'[{exp["short"]}] epoch {epoch}/{MAX_EPOCHS}: loss={train_loss:.4f} acc={train_acc:.2f}%  val_loss={val_loss:.4f} val={val_acc:.2f}%  '
              f'train={ep_train/60:.1f} min  val={ep_val/60:.1f} min')
        if val_acc > best_val_acc:
            best_val_acc, best_epoch = val_acc, epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    peak_train_mb = torch.cuda.max_memory_allocated() / 2**20 if device.type == 'cuda' else float('nan')
    del optimizer, scheduler, scaler
    return dict(history=history, best_val_acc=best_val_acc, best_epoch=best_epoch, best_state=best_state,
                train_time_s=train_time, val_time_s=val_time, peak_train_mem_mb=peak_train_mb)

def run_experiment(exp):
    t_all = time.perf_counter()
    model = build_model(exp)
    params = count_params(model)
    print(f"[{exp['short']}] parameters: total={params['total']:,}  trainable={params['trainable']:,}")
    tr = train_model(model, exp, exp['short'])
    last_learned = learned_params(model)
    model.load_state_dict(tr['best_state'])                 # best-validation weights, exactly as in the baseline test cell
    best_learned = learned_params(model)
    gc.collect()
    if device.type == 'cuda': torch.cuda.empty_cache()

    test_acc = {shot: evaluate_per_task(test_loaders[shot], model, desc=f'Testing {shot}-shot') for shot in EVAL_SHOTS}
    for shot in EVAL_SHOTS:
        print(f'[{exp["short"]}] TEST 5-way {shot}-shot: {test_acc[shot].mean():.2f}% over {len(test_acc[shot])} fixed episodes')

    infer = {shot: benchmark_inference(model, test_loaders[shot]) for shot in EVAL_SHOTS}
    gflops = {shot: measure_gflops(model, test_loaders[shot]) for shot in EVAL_SHOTS}
    cross_diag = collect_cross_diagnostics(model, test_loaders[max(EVAL_SHOTS)]) if cross_modules(model) else []
    self_attn_summary = [dict(stage=i + 1, q_conv=conv_summary(m.query_conv), k_conv=conv_summary(m.key_conv), v_conv=conv_summary(m.value_conv))
                         for i, m in enumerate(model.attention_modules)]
    res = dict(key=exp['key'], name=exp['name'], short=exp['short'], params=params, best_val_acc=tr['best_val_acc'],
               best_epoch=tr['best_epoch'], history=tr['history'], train_time_s=tr['train_time_s'], val_time_s=tr['val_time_s'],
               peak_train_mem_mb=tr['peak_train_mem_mb'], test_acc_per_episode={int(k): v.tolist() for k, v in test_acc.items()},
               inference={int(k): v for k, v in infer.items()}, gflops_per_episode={int(k): v for k, v in gflops.items()},
               learned_best=best_learned, learned_last=last_learned, cross_diagnostics=cross_diag,
               self_attention_weight_summary=self_attn_summary, wall_time_s=time.perf_counter() - t_all)
    del model, tr
    gc.collect()
    if device.type == 'cuda': torch.cuda.empty_cache()
    return res

print('Utilities ready.')

In [ ]:
# ============================== Sanity checks (fast) ==============================
print('--- 1. Shared parameters are initialised identically in all three models ---')
_models = {e['key']: build_model(e) for e in EXPERIMENTS}
_base_sd = _models['baseline'].state_dict()
for k, m in _models.items():
    sd = m.state_dict()
    shared = [n for n in _base_sd if n in sd]
    assert len(shared) == len(_base_sd)
    assert all(torch.equal(_base_sd[n], sd[n]) for n in shared), f'{k}: shared parameters differ from baseline!'
    print(f'{k:14s} OK  (extra parameters: {sum(p.numel() for n, p in m.named_parameters() if n not in _base_sd):,})')

print('\n--- 2. Shapes, residual grids and gradient flow ---')
way, shot, nq = 5, 2, 3
s = torch.randn(way * shot, 3, IMAGE_SIZE, IMAGE_SIZE, device=device)
sl = torch.arange(way, device=device).repeat_interleave(shot)
q = torch.randn(way * nq, 3, IMAGE_SIZE, IMAGE_SIZE, device=device)
ql = torch.arange(way, device=device).repeat_interleave(nq)
for k, m in _models.items():
    m.train(); m.zero_grad()
    out = m(s, sl, q)
    assert out.shape == (way * nq, way), out.shape
    nn.functional.cross_entropy(out, ql).backward()
    for n, p in m.named_parameters():
        if n.startswith('cross_'):
            assert p.grad is not None and torch.isfinite(p.grad).all(), f'no gradient for {n}'
    print(f'{k:14s} output {tuple(out.shape)} OK, cross-attention parameters receive gradients: {len(cross_modules(m)) > 0}')

print('\n--- 3. With cross-attention gates = 0 the cross models reproduce the baseline exactly ---')
_models['baseline'].eval()
with torch.no_grad():
    ref = _models['baseline'](s, sl, q)
    for k in ('adjacent', 'bidirectional'):
        m = _models[k]; m.load_state_dict(_base_sd, strict=False); m.eval()
        for _, cm in cross_modules(m): cm.gamma.zero_()
        diff = (m(s, sl, q) - ref).abs().max().item()
        assert diff < 1e-4, f'{k}: max diff {diff}'
        print(f'{k:14s} max |score difference| vs baseline = {diff:.2e}  OK')

print('\n--- 4. Cross-attention links ---')
tab = []
for k in ('adjacent', 'bidirectional'):
    for n, cm in cross_modules(_models[k]):
        tab.append(dict(model=k, link=cm.label, q_ch=cm.query_conv.in_channels, kv_ch=cm.key_conv.in_channels, d_qk=cm.d,
                        params=sum(p.numel() for p in cm.parameters())))
print(pd.DataFrame(tab).to_string(index=False))
del _models, s, q
gc.collect()
if device.type == 'cuda': torch.cuda.empty_cache()
print('\nAll sanity checks passed.')

In [ ]:
# ============================== Run the three experiments ==============================
run_config = dict(seed=SEED, eval_seed=EVAL_SEED, image_size=IMAGE_SIZE, max_epochs=MAX_EPOCHS, train_episodes_per_epoch=N_TRAINING_EPISODES,
                  val_tasks=N_VALIDATION_TASKS, test_tasks=N_TEST_TASKS, train_way=TRAIN_WAY, test_way=TEST_WAY, train_shot=SHOT, query=QUERY,
                  eval_shots=list(EVAL_SHOTS), distance_metric=DISTANCE_METRIC, amp=AMP, backbone_lr=backbone_lr, weights_lr=weights_lr,
                  gamma_init=gamma_init, cross_gamma_init=CROSS_GAMMA_INIT, stage_nodes=list(STAGE_NODES), stage_weights_init=list(STAGE_WEIGHTS),
                  pretrained_backbone=USE_IMAGENET_PRETRAINED, torch=torch.__version__, python=platform.python_version(),
                  gpu=torch.cuda.get_device_name(0) if device.type == 'cuda' else 'cpu')
json.dump(run_config, open(RESULTS_DIR / 'run_config.json', 'w'), indent=2)

results = {}
t_start = time.time()
outer = tqdm(EXPERIMENTS, desc='Experiments', unit='model')
for idx, exp in enumerate(outer, 1):
    outer.set_postfix_str(f"{idx}/{len(EXPERIMENTS)}: {exp['short']}")
    print('\n' + '=' * 78 + f"\nEXPERIMENT {idx}/{len(EXPERIMENTS)}: {exp['name']}\n" + '=' * 78)
    results[exp['key']] = run_experiment(exp)
    json.dump(results[exp['key']], open(RESULTS_DIR / f"raw_results_{exp['key']}.json", 'w'), indent=1)
    done = time.time() - t_start
    print(f"[{exp['short']}] finished in {results[exp['key']]['wall_time_s']/60:.1f} min  |  elapsed {done/60:.1f} min  |  "
          f"est. remaining {(done/idx)*(len(EXPERIMENTS)-idx)/60:.1f} min")
print(f'\nAll experiments finished in {(time.time()-t_start)/60:.1f} min.')

In [ ]:
# ============================== Comparison tables (CSV) ==============================
try:
    from scipy import stats as _st
except Exception:
    _st = None

order = [e['key'] for e in EXPERIMENTS]
short = {e['key']: e['short'] for e in EXPERIMENTS}
full  = {e['key']: e['name'] for e in EXPERIMENTS}
ci95 = lambda a: 1.96 * np.std(a, ddof=1) / np.sqrt(len(a)) if len(a) > 1 else float('nan')
acc = {k: {s: np.array(results[k]['test_acc_per_episode'][s]) for s in EVAL_SHOTS} for k in order}

# ---- Table 1: accuracy (mean over fixed episodes, 95% CI, paired difference vs baseline on the SAME episodes)
rows = []
for k in order:
    for s in EVAL_SHOTS:
        a, b = acc[k][s], acc['baseline'][s]
        d = a - b
        p = float(_st.ttest_rel(a, b).pvalue) if (_st is not None and k != 'baseline' and len(a) > 1) else float('nan')
        rows.append({'model': full[k], 'setting': f'5-way {s}-shot', 'test_acc_%': round(a.mean(), 2), 'ci95_%': round(ci95(a), 2),
                     'n_test_episodes': len(a), 'delta_vs_baseline_%': round(d.mean(), 2) if k != 'baseline' else 0.0,
                     'delta_ci95_%': round(ci95(d), 2) if k != 'baseline' else 0.0,
                     'paired_t_test_p': round(p, 4) if p == p else float('nan'),
                     'best_val_acc_5shot_%': round(results[k]['best_val_acc'], 2), 'best_epoch': results[k]['best_epoch']})
table1 = pd.DataFrame(rows)

# ---- Table 2: efficiency
rows = []
for k in order:
    r = results[k]; p = r['params']; n_ep = len(r['history'])
    row = {'model': full[k], 'total_params': p['total'], 'trainable_params': p['trainable'], 'total_params_M': round(p['total'] / 1e6, 3),
           'train_time_total_min': round(r['train_time_s'] / 60, 2), 'train_time_per_epoch_min': round(r['train_time_s'] / 60 / n_ep, 2),
           'validation_time_total_min': round(r['val_time_s'] / 60, 2), 'peak_train_gpu_mem_MB': round(r['peak_train_mem_mb'], 0)}
    for s in EVAL_SHOTS:
        i = r['inference'][s]
        row.update({f'infer_ms_per_episode_{s}shot': round(i['ms_per_episode_mean'], 2), f'infer_ms_per_query_img_{s}shot': round(i['ms_per_query_image'], 3),
                    f'infer_query_img_per_s_{s}shot': round(i['query_images_per_s'], 1), f'peak_infer_gpu_mem_MB_{s}shot': round(i['peak_gpu_mem_mb'], 0),
                    f'GFLOPs_per_episode_{s}shot': round(r['gflops_per_episode'][s], 3)})
    row['wall_time_incl_test_and_benchmarks_min'] = round(r['wall_time_s'] / 60, 2)
    rows.append(row)
table2 = pd.DataFrame(rows)
b = table2.iloc[0]
for col in ('total_params', 'train_time_total_min', f'infer_ms_per_episode_{max(EVAL_SHOTS)}shot', 'peak_train_gpu_mem_MB'):
    table2[f'{col}_x_baseline'] = (table2[col] / b[col]).round(3)

# ---- Table 3: parameter breakdown
table3 = pd.DataFrame([{'model': full[k], **{c: results[k]['params'][c] for c in
                        ('total', 'trainable', 'backbone', 'self_attention', 'cross_attention_convs', 'cross_attention_gates', 'scalars')}} for k in order])

# ---- Table 4: training history
table4 = pd.concat([pd.DataFrame(results[k]['history']) for k in order], ignore_index=True)

# ---- Table 5: learned stage weights and gamma values
rows = []
for k in order:
    for which, key in (('best-validation checkpoint (used for test)', 'learned_best'), ('last epoch', 'learned_last')):
        for name, val in results[k][key].items():
            rows.append({'model': full[k], 'weights_from': which, 'parameter': name, 'value': round(val, 6)})
table5 = pd.DataFrame(rows)

# ---- Table 6: cross-attention diagnostics (compact)
rows = []
for k in order:
    for d in results[k]['cross_diagnostics']:
        rows.append({'model': full[k], **{kk: (round(v, 5) if isinstance(v, float) else v) for kk, v in d.items() if not kk.endswith('_conv')},
                     **{f'{c}_{m}': round(d[f'{c}_conv'][m], 5) for c in ('q', 'k', 'v') for m in ('fro_norm', 'mean_abs', 'std')}})
table6 = pd.DataFrame(rows)

# ---- Table 7: per-episode accuracies (same episodes for every model)
table7 = pd.DataFrame({'episode': np.arange(1, N_TEST_TASKS + 1)})
for k in order:
    for s in EVAL_SHOTS:
        table7[f'{short[k]}_{s}shot'] = np.round(acc[k][s], 3)

for name, df in [('table1_accuracy_comparison', table1), ('table2_efficiency_comparison', table2), ('table3_parameter_breakdown', table3),
                 ('table4_training_history', table4), ('table5_learned_stage_weights_and_gammas', table5),
                 ('table6_cross_attention_diagnostics', table6), ('table7_per_episode_test_accuracy', table7)]:
    df.to_csv(RESULTS_DIR / f'{name}.csv', index=False)

# compact per-model weight/statistics reports (JSON)
for k in order:
    rep = dict(model=full[k], learned_params_best_checkpoint=results[k]['learned_best'], learned_params_last_epoch=results[k]['learned_last'],
               self_attention_weight_summary=results[k]['self_attention_weight_summary'], cross_attention=results[k]['cross_diagnostics'])
    json.dump(rep, open(RESULTS_DIR / f'learned_params_and_stats_{k}.json', 'w'), indent=1)

pd.set_option('display.width', 250); pd.set_option('display.max_columns', 50)
print('TABLE 1 - Test accuracy (fixed episodes, 95% CI; delta = paired difference vs baseline on identical episodes)')
display(table1)
print('TABLE 2 - Efficiency'); display(table2.T)
print('TABLE 3 - Parameter breakdown'); display(table3)
print('TABLE 5 - Learned stage weights and gammas (best checkpoint)'); display(table5[table5['weights_from'].str.startswith('best')])
if len(table6): print('TABLE 6 - Cross-attention diagnostics'); display(table6.drop(columns=['module']))
print('Note: each model was trained once (seed 42). The 95% CIs reflect episode sampling only, not training-seed variance; '
      'differences of a few tenths of a percent should not be over-interpreted.')

In [ ]:
# ============================== Plots ==============================
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.3, 'axes.spines.top': False, 'axes.spines.right': False})
col = [COLORS[k] for k in order]; lab = [short[k] for k in order]

# Fig 1: test accuracy
fig, ax = plt.subplots(figsize=(8, 4.8))
w = 0.26; x = np.arange(len(EVAL_SHOTS))
for i, k in enumerate(order):
    means = [acc[k][s].mean() for s in EVAL_SHOTS]; errs = [ci95(acc[k][s]) for s in EVAL_SHOTS]
    bars = ax.bar(x + (i - 1) * w, means, w, yerr=errs, capsize=4, color=COLORS[k], label=short[k])
    for bx, m in zip(bars, means): ax.text(bx.get_x() + bx.get_width() / 2, m + 0.6, f'{m:.2f}', ha='center', fontsize=8)
allm = [acc[k][s].mean() for k in order for s in EVAL_SHOTS]
ax.set_ylim(min(allm) - 8, max(allm) + 4); ax.set_xticks(x); ax.set_xticklabels([f'5-way {s}-shot' for s in EVAL_SHOTS])
ax.set_ylabel('Test accuracy (%)'); ax.set_title('MiniImageNet test accuracy (600 fixed episodes, 95% CI)'); ax.legend()
fig.tight_layout(); fig.savefig(RESULTS_DIR / 'fig1_test_accuracy.png', dpi=150); plt.show()

# Fig 2 (four separate plots): training accuracy, training loss, validation accuracy, validation loss per epoch
for col_name, title, ylabel, fname in (
        ('train_acc',  'Training accuracy per epoch (running, over training episodes)', 'accuracy (%)',  'fig2a_training_accuracy.png'),
        ('train_loss', 'Training loss per epoch',                                       'cross-entropy', 'fig2b_training_loss.png'),
        ('val_acc',    'Validation accuracy per epoch (5-way 5-shot, fixed episodes)',  'accuracy (%)',  'fig2c_validation_accuracy.png'),
        ('val_loss',   'Validation loss per epoch (5-way 5-shot, fixed episodes)',      'cross-entropy', 'fig2d_validation_loss.png')):
    fig, ax = plt.subplots(figsize=(6.4, 4.4))
    for k in order:
        h = pd.DataFrame(results[k]['history'])
        ax.plot(h['epoch'], h[col_name], marker='o', color=COLORS[k], label=short[k])
    ax.set_title(title, fontsize=10); ax.set_xlabel('epoch'); ax.set_ylabel(ylabel); ax.legend()
    fig.tight_layout(); fig.savefig(RESULTS_DIR / fname, dpi=150); plt.show()

# Fig 3 (separate plots): parameters, training time, inference time, GPU memory
S = max(EVAL_SHOTS)
def bar_plot(title, ylabel, vals, fname, fmt):
    fig, ax = plt.subplots(figsize=(5.6, 4.2))
    bars = ax.bar(lab, vals, color=col)
    for bx, v in zip(bars, vals): ax.text(bx.get_x() + bx.get_width() / 2, v, fmt.format(v) if v == v else 'n/a', ha='center', va='bottom', fontsize=9)
    ax.set_title(title); ax.set_ylabel(ylabel); ax.margins(y=0.15)
    fig.tight_layout(); fig.savefig(RESULTS_DIR / fname, dpi=150); plt.show()

bar_plot('Total parameters', 'millions', [results[k]['params']['total'] / 1e6 for k in order], 'fig3a_parameters.png', '{:.2f}')
bar_plot(f'Training time ({MAX_EPOCHS} epochs)', 'minutes', [results[k]['train_time_s'] / 60 for k in order], 'fig3b_training_time.png', '{:.1f}')
bar_plot(f'Inference latency (5-way {S}-shot episode)', 'ms / episode', [results[k]['inference'][S]['ms_per_episode_mean'] for k in order], 'fig3c_inference_time.png', '{:.1f}')
fig, ax = plt.subplots(figsize=(6.4, 4.2)); xg = np.arange(len(order)); wg = 0.36
tr_mem = [results[k]['peak_train_mem_mb'] for k in order]; in_mem = [results[k]['inference'][S]['peak_gpu_mem_mb'] for k in order]
for off, vals, name, c in ((-wg / 2, tr_mem, 'training (peak)', '#6a6a6a'), (wg / 2, in_mem, f'inference ({S}-shot episode, peak)', '#b5b5b5')):
    bars = ax.bar(xg + off, vals, wg, label=name, color=c)
    for bx, v in zip(bars, vals): ax.text(bx.get_x() + bx.get_width() / 2, v, f'{v:.0f}' if v == v else 'n/a', ha='center', va='bottom', fontsize=8)
ax.set_xticks(xg); ax.set_xticklabels(lab); ax.set_ylabel('MB (torch allocated)'); ax.set_title('GPU memory'); ax.legend(fontsize=8); ax.margins(y=0.15)
fig.tight_layout(); fig.savefig(RESULTS_DIR / 'fig3d_gpu_memory.png', dpi=150); plt.show()

# Fig 4: accuracy vs cost
fig, axs = plt.subplots(1, 2, figsize=(11, 4.4))
for k in order:
    r = results[k]; a = acc[k][S].mean()
    axs[0].scatter(r['inference'][S]['ms_per_episode_mean'], a, s=r['params']['total'] / 1e5 * 6, color=COLORS[k], alpha=0.8, label=short[k])
    axs[1].scatter(r['gflops_per_episode'][S], a, s=r['params']['total'] / 1e5 * 6, color=COLORS[k], alpha=0.8, label=short[k])
axs[0].set_xlabel(f'Inference latency (ms / {S}-shot episode)'); axs[1].set_xlabel(f'GFLOPs / {S}-shot episode')
for a_ in axs: a_.set_ylabel(f'5-way {S}-shot test accuracy (%)'); a_.legend(loc='best'); a_.margins(0.25)
fig.suptitle('Accuracy vs. cost (marker size ~ parameter count)'); fig.tight_layout(); fig.savefig(RESULTS_DIR / 'fig4_accuracy_vs_cost.png', dpi=150); plt.show()

# Fig 5: learned weights and gammas
fig, axs = plt.subplots(1, 2, figsize=(12, 4.4), gridspec_kw={'width_ratios': [1, 1.4]})
nst = len(STAGE_NODES); xs = np.arange(nst)
for i, k in enumerate(order):
    lp = results[k]['learned_best']
    axs[0].bar(xs + (i - 1) * w, [lp[f'stage_weight_{j+1}'] for j in range(nst)], w, color=COLORS[k], label=short[k])
axs[0].plot(xs, STAGE_WEIGHTS, 'k--', lw=1, label='initial'); axs[0].set_xticks(xs); axs[0].set_xticklabels([f'Stage {j+1}' for j in range(nst)])
axs[0].set_title('Learned stage weights'); axs[0].legend(fontsize=8)
names = ['gamma1_support_selfattn', 'gamma2_query_selfattn'] + sorted({n for k in order for n in results[k]['learned_best'] if n.startswith('cross_gate')})
xs2 = np.arange(len(names))
for i, k in enumerate(order):
    axs[1].bar(xs2 + (i - 1) * w, [results[k]['learned_best'].get(n, np.nan) for n in names], w, color=COLORS[k], label=short[k])
axs[1].axhline(gamma_init, color='k', ls='--', lw=1)
axs[1].set_xticks(xs2); axs[1].set_xticklabels([n.replace('cross_gate[', '').replace(']', '').replace('_selfattn', '').replace('Stage', 'S') for n in names], rotation=60, ha='right', fontsize=8)
axs[1].set_title('Learned gamma / cross-attention gates (dashed = init 0.2)')
fig.tight_layout(); fig.savefig(RESULTS_DIR / 'fig5_learned_weights_and_gammas.png', dpi=150); plt.show()

# Fig 6 (extra): distribution of per-episode test accuracy
fig, ax = plt.subplots(figsize=(6.4, 4.4))
bp = ax.boxplot([acc[k][S] for k in order], patch_artist=True, showmeans=True)
ax.set_xticklabels(lab)
for patch, c in zip(bp['boxes'], col): patch.set_facecolor(c); patch.set_alpha(0.7)
ax.set_ylabel('accuracy per episode (%)'); ax.set_title(f'Per-episode test accuracy (5-way {S}-shot, {len(acc[order[0]][S])} episodes; triangle = mean)')
fig.tight_layout(); fig.savefig(RESULTS_DIR / 'fig6_per_episode_accuracy_distribution.png', dpi=150); plt.show()

# Fig 7 (extra): cross-attention diagnostics per link
if len(table6):
    links = list(dict.fromkeys(table6['link'])); xl = np.arange(len(links))
    fig, axs = plt.subplots(1, 2, figsize=(12.5, 4.6))
    for ax, colname, title in ((axs[0], 'attn_entropy_normalized', 'Attention entropy (1 = uniform, 0 = peaked)'),
                               (axs[1], 'update_ratio_norm', 'Update ratio  ||gamma * CA|| / ||feature||')):
        for i, k in enumerate(('adjacent', 'bidirectional')):
            sub = table6[table6['model'] == full[k]].set_index('link')[colname]
            ax.bar(xl + (i - 0.5) * 0.38, [sub.get(l, np.nan) for l in links], 0.38, color=COLORS[k], label=short[k])
        ax.set_xticks(xl); ax.set_xticklabels([l.replace('Stage', 'S') for l in links], rotation=45, ha='right', fontsize=8)
        ax.set_title(title, fontsize=10); ax.legend(fontsize=8)
    fig.tight_layout(); fig.savefig(RESULTS_DIR / 'fig7_cross_attention_diagnostics.png', dpi=150); plt.show()

In [ ]:
# ============================== ZIP + automatic download ==============================
readme = f"""MSENet vs Adjacent vs Bidirectional Cross-Attention - MiniImageNet
Config: {MAX_EPOCHS} epochs x {N_TRAINING_EPISODES} episodes, {TRAIN_WAY}-way {SHOT}-shot {QUERY}-query training, {DISTANCE_METRIC} distance, seed {SEED}.
Test: {N_TEST_TASKS} fixed 5-way episodes per shot (5-way 5-shot), identical for all models, using the best-validation checkpoint.
Files
  table1_accuracy_comparison.csv            test accuracy, 95% CI, paired difference vs baseline
  table2_efficiency_comparison.csv          params, training time, inference time, FLOPs, GPU memory
  table3_parameter_breakdown.csv            parameter counts per component
  table4_training_history.csv               per-epoch loss / val accuracy / time / learned weights
  table5_learned_stage_weights_and_gammas.csv
  table6_cross_attention_diagnostics.csv    per-link gate, attention entropy / max weight, update ratio, projection weight norms
  table7_per_episode_test_accuracy.csv
  learned_params_and_stats_<model>.json     compact weight/statistics reports
  raw_results_<model>.json, run_config.json
  fig1..fig7 *.png (accuracy, training/validation curves, efficiency, accuracy-vs-cost, learned weights, per-episode distribution, cross-attention diagnostics)
No model checkpoints are included.
"""
(RESULTS_DIR / 'README.txt').write_text(readme)

if ZIP_PATH.exists(): ZIP_PATH.unlink()
with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(RESULTS_DIR.rglob('*')):
        if p.is_file():
            assert p.suffix not in ('.pth', '.pt', '.ckpt'), f'checkpoint found: {p}'
            zf.write(p, arcname=f'msenet_results/{p.relative_to(RESULTS_DIR)}')
print(f'ZIP created: {ZIP_PATH}  ({ZIP_PATH.stat().st_size / 1e6:.2f} MB)')
with zipfile.ZipFile(ZIP_PATH) as zf:
    for i in zf.infolist(): print(f'  {i.filename}  ({i.file_size/1e3:.1f} KB)')

from IPython.display import FileLink, HTML, display
link = FileLink(os.path.relpath(ZIP_PATH), result_html_prefix='Download results ZIP: ')
display(HTML(link._repr_html_() +
             "<script>try{var a=document.currentScript.parentElement.querySelector('a');if(a){a.setAttribute('download','');a.click();}}catch(e){}</script>"))
print('If the browser did not start the download, click the link above, or use the notebook Output tab: ', ZIP_PATH)